In [1]:
!pip install -q peft
!pip install -q tqdm

In [3]:
import pandas as pd

In [5]:
#Load the dataset
reviews_df = pd.read_csv("C:/Users/hp/Documents/Res/Jupyter/Finetuned LLM/imdb_reviews_dataset/IMDB Dataset.csv")

reviews_df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
#Construct a dataframe( Sample 100 examples of each class. )
positive_samples = reviews_df[ reviews_df["sentiment"] == "positive" ].sample( n = 100, random_state = 42 )
negative_samples = reviews_df[ reviews_df["sentiment"] == "negative" ].sample( n = 100, random_state = 42 )

df = pd.concat( [ positive_samples, negative_samples ], ignore_index = True )

df

,review,sentiment
0,I don't know how or why this film has a meager...,positive
1,For a long time it seemed like all the good Ca...,positive
2,Terry Gilliam's and David Peoples' teamed up t...,positive
3,What is there to say about an anti-establishme...,positive
4,This movie was made only 48 years after the en...,positive
...,...,...
195,This movie was on the pay channels today and I...,negative
196,This film has got so much in it. Prehistoric s...,negative
197,Telly Savalas put on a passable (but no better...,negative
198,Well. Where to begin. Let's just say this; avo...,negative


In [9]:
#Dataframe is currently perfectly ordered with 200 positive and 200 negative samples.
#Shuffle the dataframe so that the model sees different labels on each iteration of the training loop.
df = df.sample( frac = 1, random_state = 42 ).reset_index( drop = True )

df

,review,sentiment
0,I would like to tell you just a few things bef...,positive
1,This is a VERY underrated movie to say the lea...,positive
2,Du Rififi Chez Les Hommes/Rififi(1955) can on ...,positive
3,"""Beowulf"" is like a very bad game : no charact...",negative
4,It seems on the surface to be a romantic _plan...,negative
...,...,...
195,If I wouldn't have had any expectations of thi...,negative
196,I got this movie from eBay mainly because I'm ...,positive
197,"Chang Cheh's ""Shaolin Temple"" might very well ...",positive
198,I must say that I am fairly disappointed by th...,negative


In [11]:
#For tracking the progress of the training loop.
from tqdm import tqdm

In [13]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

In [15]:
#Load the model.
device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained( model_id, dtype = torch.float32 ).to( device )

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [17]:
#Load tokenizer
tokenizer = AutoTokenizer.from_pretrained( model_id )

In [19]:
#Create PEFT congif for the model
peft_config = LoraConfig(
    r = 16,
    lora_alpha = 32,
    task_type = TaskType.CAUSAL_LM
)

In [21]:
#Get a model with PEFT
model = get_peft_model( model, peft_config )

model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [23]:
#Choose optimizer
optimizer = torch.optim.AdamW( model.parameters(), lr = 2e-4 )

In [27]:
#Train the model
EPOCHS = 2
MAX_CHARS = 600
all_losses = []

for epoch in range( EPOCHS ):
    model.train()
    losses = []

    #Normally iterating over the zip is enough, but to keep track of the progress of the training loop, wrapping it with tqdm helps.
    for review, sentiment in tqdm( zip( df["review"], df["sentiment"] ), total = len(df), desc = f"Epoch: {epoch + 1}" ):
        review = review[:MAX_CHARS]

        #Prompt is what model sees, input.
        prompt = f"Review: {review}\nSentiment:"
        #Text is what the model should return, output.
        text = f"{prompt} {sentiment}{tokenizer.eos_token}"

        #Tokenizer converts the character strings into tokens that the model understands. 
        #The output will be a vector of numbers(tokens), which are indices to words in the model's vocabulary.
        inputs = tokenizer( text, return_tensors = 'pt' ).to( model.device )
        prompt = tokenizer( prompt, return_tensors = 'pt' ).to( model.device )

        #["input_ids"] actually returns the vector of tokens.
        #.shape gives (1 X T) where T is the number of tokens.
        #.shape[1] retrives the variable T.
        prompt_len = prompt["input_ids"].shape[1]

        #Clone the token vector of inputs so that the next line does not modify the original.
        labels = inputs["input_ids"].clone()
        #These labels will be used to compute the loss and grade the output of the LLM.
        #By setting the tokens up until prompt length to a negative number, the grader does not calculate loss for it and ignores it.
        #This is desirable since we only want to grade the LLM the generated output, not the prompt in itself.
        labels[:, :prompt_len] = -100

        #Pass the model the inputs so that it can train.
        #Passing labels gives the expected output tokens for each word generated.
        outputs = model( **inputs, labels = labels )

        #Computes the loss.
        loss = outputs.loss

        #Backpropagation.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append( loss.item() )
        all_losses.append( loss.item() )

    average_loss = sum(losses) / len(losses)
    print(f"Epoch: {epoch + 1} and loss: {average_loss}.")
    

Epoch: 1: 100%|██████████████████████████████████████████████████████████████████████| 200/200 [06:21<00:00,  1.91s/it]


Epoch: 1 and loss: 0.07389821717049927.


Epoch: 2: 100%|██████████████████████████████████████████████████████████████████████| 200/200 [06:25<00:00,  1.93s/it]

Epoch: 2 and loss: 0.06954982760624262.


In [33]:
#Save the trained model.
model.save_pretrained( "qwen2.5-0.5b-lora" )

In [43]:
#Import the saved model and use it.
from peft import PeftModel

device = "cpu"

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained( model_id, dtype = torch.float32 ).to( device )
model = PeftModel.from_pretrained( model, "qwen2.5-0.5b-lora" )

review = "That movie was interesting."

text = f"Review: {review}\nSentiment:"

inputs = tokenizer( text, return_tensors = 'pt' ).to( model.device )

outputs = model.generate( **inputs.to( device ), max_new_tokens = 20 )

decoded_output = tokenizer.decode( outputs[0], skip_special_tokens = True )

decoded_output

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

'Review: That movie was interesting.\nSentiment: positive'